# 下载模型

In [1]:
# import os
# os.environ['MODELSCOPE_DOMAIN'] = 'www.modelscope.ai'
# #Model Download
# from modelscope import snapshot_download
# model_dir = snapshot_download('vllm-ascend/Llama-2-7b-hf', cache_dir = '/root/autodl-tmp', revision = 'master')

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_path = "/root/Midi-Tuning/pretrained/Llama-2-7b-hf"

try:
    print("正在验证分词器...")
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    
    print("正在验证模型权重结构（仅加载 Meta 数据）...")
    # device_map="meta" 可以快速检查结构而不占用显存
    model = AutoModelForCausalLM.from_pretrained(
        model_path, 
        device_map="meta", 
        local_files_only=True,
        torch_dtype=torch.float16
    )
    print("✅ 验证通过：模型结构完整且可识别。")
except Exception as e:
    print(f"❌ 验证失败：{e}")

正在验证分词器...
正在验证模型权重结构（仅加载 Meta 数据）...
[2026-03-19 20:06:21,507] [INFO] [real_accelerator.py:158:get_accelerator] Setting ds_accelerator to cuda (auto detect)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

✅ 验证通过：模型结构完整且可识别。


# 训练

In [1]:
deepspeed --master_port=29600 --include="localhost:0" src/midituning/finetune.py \
    --model_name_or_path pretrained/Llama-2-7b-hf \
    --data_path data/light/data_fmt_dialog/train.json \
    --weight_beta 1.0 \
    --max_instruction_length 256 \
    --max_utterance_length 72 \
    --max_rounds 10 \
    --num_proc 8 \
    --output_dir /root/autodl-tmp/Output/midi-Llama \
    --per_device_train_batch_size 1 \
    --per_device_eval_batch_size 1 \
    --gradient_accumulation_steps 16 \
    --num_train_epochs 3 \
    --evaluation_strategy "no" \
    --learning_rate 2e-5 \
    --warmup_ratio 0.03 \
    --lr_scheduler_type "cosine" \
    --logging_steps 1 \
    --save_strategy "steps" \
    --save_steps 100 \
    --save_total_limit 3 \
    --lora_r 8 \
    --lora_alpha 16 \
    --lora_dropout 0.05 \
    --q_lora True \
    --bf16 True \
    --deepspeed config/deepspeed_config_c2.json

SyntaxError: invalid decimal literal (3912267807.py, line 2)

# 推理

In [1]:
export CUDA_VISIBLE_DEVICES="0"

accelerate launch --main_process_port 29600 \
    --num_processes=1 \
    --num_machines=1 \
    --mixed_precision=bf16 \
    --dynamo_backend=no \
    src/midituning/generate.py \
    --model_path /root/autodl-tmp/Output/midi-Llama \
    --bf16 True \
    --test_data_path data/light/data_fmt_dialog/test.json \
    --test_unseen_data_path data/light/data_fmt_dialog/test_unseen.json \
    --output_dir /root/autodl-tmp/Output/midi-Llama\
    --max_instruction_length 320 \
    --max_utterance_length 100 \
    --max_rounds 10 \
    --max_new_tokens 100 \
    --temperature 0.5 \
    --top_p 0.75 \
    --top_k 40

SyntaxError: invalid syntax (2796551346.py, line 1)

# 评估

In [ ]:
# 以 LIGHT 数据集为例
python eval/eval_light.py \
    --eval_file /root/autodl-tmp/Output/midi-Llama/test_output.jsonl \
    --gold_file data/light/light_test.jsonl 

python eval/eval_light.py \
    --eval_file results/light/midi_${MODEL_NAME}/test_unseen_output.jsonl \
    --gold_file data/light/light_test_unseen.jsonl

In [ ]:
# 训练评估器
python src/detector/run.py --data_dir data/${DATASET_NAME} \
    --output_dir logs/${DATASET_NAME}/detector \
    --bert_model pretrained/bert-base-uncased \
    --architecture "detect" \
    --max_length 500 \
    --train_batch_size 32 \
    --eval_batch_size 32 \
    --learning_rate 2e-5 \
    --warmup_steps 500

# 进行评估
python src/detector/run.py --eval --plot --data_dir data/${DATASET_NAME} \
    --output_dir logs/${DATASET_NAME}/detector \
    --bert_model pretrained/bert-base-uncased \
    --max_length 500 \
    --eval_batch_size 32

In [ ]:
# GPT-4 Score
# 以 LIGHT 数据集为例
python eval/eval_by_gpt.py \
    --eval_file /root/autodl-tmp/Output/midi-Llama/test_output.jsonl \
    --gold_file data/light/light_test.jsonl \
    --prompt_template prompt/eval_light.txt \
    --model "deepseek-ai/DeepSeek-V3" \
    --api_key_file /root/Midi-Tuning/eval/openai_api_key.txt

In [ ]:
# 原版
# python eval/eval_by_gpt.py \
#     --eval_file results/light/midi_${MODEL_NAME}/test_output.jsonl \
#     --gold_file data/light/light_test.jsonl \
#     --prompt_template prompt/eval_light.txt \
#     --model "gpt-4-turbo"

# 部署与推理

In [ ]:
# vllm
vllm serve autodl-tmp/vllm-ascend/Llama-2-7b-hf/ \
    --port 6006 \
    --host 0.0.0.0 \
    --enable-lora \
    --lora-modules my-lora=<你的LoRA适配器路径> \
    --max-lora-rank 16 \
    --dtype auto